# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AnanyaGujral/flyrank-ml-internship-start/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

Runs offline on the bundled slice. The contract below is stated first, then **every claim gets a check next to it**.

## 1. Unit of analysis + time window

- **One row = one pseudonymized content page**, 30,000 pages across 32 clients.
- **The slice is a single trailing-90-day snapshot** that also stores two consecutive 30-day windows:
  `*_prev_30d` (days 31–60) and `*_last_30d` (days 1–30).
- **My decision window is the prior-30 window; my outcome window is the later-30 window.** The gap
  between them is what makes the label forward-looking rather than circular.

In [1]:
# ---- shared offline setup: bundled slice, leak-free pool + features, transparent rule ----
import json
from pathlib import Path
import numpy as np
import pandas as pd

SEED = 42
MIN_PREV_IMPRESSIONS = 100     # "already visible" floor on the TRAILING window
DROP_FACTOR = 0.8              # declined = next-30 impressions < 80% of prior-30

root = Path.cwd().resolve()
while root != root.parent and not (root / "AGENTS.md").exists():
    root = root.parent
DATA = root / "data" / "raw" / "content_refresh_anonymized.csv"
OUT = root / "work" / "outputs"
OUT.mkdir(parents=True, exist_ok=True)
raw = pd.read_csv(DATA)


def build_pool(frame):
    """Already-visible pages with a usable trailing window, plus leak-free features.

    Every feature is measured in the PRIOR 30-day window (days 31-60) or is static
    metadata. Nothing is taken from the 90-day totals or the outcome window, because
    those overlap the label's comparison window.
    """
    pool = frame[frame["impressions_prev_30d"] >= MIN_PREV_IMPRESSIONS].copy()
    pool["declined_next_30d"] = (
        pool["impressions_last_30d"] < DROP_FACTOR * pool["impressions_prev_30d"]).astype(int)
    pool["log_impr_prev"] = np.log1p(pool["impressions_prev_30d"])
    pool["log_clk_prev"] = np.log1p(pool["clicks_prev_30d"])
    pool["log_sess_prev"] = np.log1p(pool["sessions_prev_30d"])
    pool["ctr_prev"] = 100 * pool["clicks_prev_30d"] / pool["impressions_prev_30d"]
    pool["clk_per_sess_prev"] = pool["clicks_prev_30d"] / pool["sessions_prev_30d"].replace(0, np.nan)
    pool["has_keyword"] = pool["search_volume"].notna().astype(int)
    pool["has_clicks_prev"] = (pool["clicks_prev_30d"] > 0).astype(int)
    pool["has_sessions_prev"] = (pool["sessions_prev_30d"] > 0).astype(int)
    g = pool.groupby("client_id")
    pool["ctr_prev_z"] = ((pool["ctr_prev"] - g["ctr_prev"].transform("mean"))
                          / g["ctr_prev"].transform("std").replace(0, np.nan))
    pool["impr_prev_share"] = pool["impressions_prev_30d"] / g["impressions_prev_30d"].transform("sum")
    pool["clk_prev_share"] = pool["clicks_prev_30d"] / g["clicks_prev_30d"].transform("sum").replace(0, np.nan)
    return pool


NUM_FEATURES = ["log_impr_prev", "log_clk_prev", "log_sess_prev", "ctr_prev", "clk_per_sess_prev",
                "ctr_prev_z", "impr_prev_share", "clk_prev_share",
                "search_volume", "competition", "cpc", "word_count", "char_count",
                "content_age_days", "days_since_last_update",
                "has_keyword", "has_clicks_prev", "has_sessions_prev"]
CAT_FEATURES = ["competition_level", "content_type", "main_intent", "age_tier",
                "freshness_tier", "word_count_tier"]


def feature_matrix(pool):
    Xn = pool[NUM_FEATURES].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return pd.concat([Xn, pd.get_dummies(pool[CAT_FEATURES].astype(str), dtype=float)], axis=1)


def rulescore(pool):
    """Transparent rule: low trailing CTR, weighted by trailing exposure."""
    med = float(pool["ctr_prev"].median())
    return (pool["ctr_prev"] <= 0.5 * med).to_numpy() * np.log1p(pool["impressions_prev_30d"].to_numpy())


pool = build_pool(raw)
X = feature_matrix(pool)
y = pool["declined_next_30d"].to_numpy()
groups = pool["client_id"].to_numpy()
rule = rulescore(pool)
print(f"bundled slice: {len(raw):,} rows -> pool {len(pool):,} pages | "
      f"{pool['client_id'].nunique()} pseudo-clients")
print(f"decline base rate in pool: {y.mean():.3f} | feature matrix: {X.shape}")
print("label: next-30 impressions < 80% of prior-30 (all features from the prior-30 window + metadata)")


def pk(table_scores, y_true, ks=(10, 20, 50, 100)):
    y_true = np.asarray(y_true)
    order = np.argsort(-np.asarray(table_scores))
    return {str(k): round(float(y_true[order[:k]].mean()), 3) for k in ks}

bundled slice: 30,000 rows -> pool 18,010 pages | 30 pseudo-clients
decline base rate in pool: 0.616 | feature matrix: (18010, 43)
label: next-30 impressions < 80% of prior-30 (all features from the prior-30 window + metadata)


## 2. Fields: feature / label / context / excluded

**Feature** (knowable at the end of the prior-30 window): prior-window impressions/clicks/sessions and
their log forms, prior-window CTR and click-per-session, per-client normalised CTR, prior-window shares
of client demand, keyword context (`search_volume`, `competition`, `cpc`), content depth
(`word_count`, `char_count`), age/freshness, presence flags, and categorical descriptors.

**Label:** `declined_next_30d = impressions_last_30d < 0.8 * impressions_prev_30d`.

**Context (never a feature):** `content_id`, `client_id` — grouping/joins only.

**Excluded, with the reason:** see §3c and `w03_feature_leakage_check.ipynb`.

In [2]:
context = ["content_id", "client_id"]
excluded = {
    "trend_direction, trend_pct": "they DEFINE the label -> tautology",
    "impressions_last_30d / clicks_last_30d / sessions_last_30d": "the label's outcome window",
    "impressions_90d, clicks_90d, sessions_90d, ... (all 90d totals)": "overlap the label window (days 1-60)",
    "ctr, avg_position, engagement_rate, scroll_rate, ai_traffic_pct": "computed over the 90d window -> overlap",
    "days_with_impressions, days_with_sessions": "90d coverage -> overlap",
    "impression_tier, position_tier": "bucketed from the 90d columns above",
    "provider_used, model_used": "production/LLM metadata, not a page signal (data-dictionary rule)",
}
print("FEATURES:", len(NUM_FEATURES), "numeric +", len(CAT_FEATURES), "categorical")
print("CONTEXT :", context)
print("EXCLUDED:")
for k, why in excluded.items():
    print(f"  - {k}: {why}")

FEATURES: 18 numeric + 6 categorical
CONTEXT : ['content_id', 'client_id']
EXCLUDED:
  - trend_direction, trend_pct: they DEFINE the label -> tautology
  - impressions_last_30d / clicks_last_30d / sessions_last_30d: the label's outcome window
  - impressions_90d, clicks_90d, sessions_90d, ... (all 90d totals): overlap the label window (days 1-60)
  - ctr, avg_position, engagement_rate, scroll_rate, ai_traffic_pct: computed over the 90d window -> overlap
  - days_with_impressions, days_with_sessions: 90d coverage -> overlap
  - impression_tier, position_tier: bucketed from the 90d columns above
  - provider_used, model_used: production/LLM metadata, not a page signal (data-dictionary rule)


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a check here. A contract claim without a check next to it is a guess.*

In [3]:
# FACT 1 - the grain holds: one row really is one page.
dupes = int(raw["content_id"].duplicated().sum())
print(f"FACT 1 grain: {len(raw):,} rows, {raw['content_id'].nunique():,} unique content_id "
      f"-> duplicated ids: {dupes}")

# FACT 2 - counts and the two stacked windows.
print(f"FACT 2 windows: rows with prior-30 impressions   = {(raw['impressions_prev_30d'] > 0).sum():,}")
print(f"                rows with later-30 impressions   = {(raw['impressions_last_30d'] > 0).sum():,}")
print(f"                prior-30 impressions == 0 (no usable denominator): {(raw['impressions_prev_30d'] == 0).sum():,}")

# FACT 3 - missingness is systematic, not random (by content_type).
miss = raw.groupby("content_type").agg(
    n=("content_id", "count"),
    keyword_missing=("search_volume", lambda s: round(float(s.isna().mean()), 2)),
    word_count_missing=("word_count", lambda s: round(float(s.isna().mean()), 2)))
print("FACT 3 missingness by content_type:")
print(miss.to_string())

FACT 1 grain: 30,000 rows, 30,000 unique content_id -> duplicated ids: 0
FACT 2 windows: rows with prior-30 impressions   = 26,612
                rows with later-30 impressions   = 27,453
                prior-30 impressions == 0 (no usable denominator): 3,388
FACT 3 missingness by content_type:
                        n  keyword_missing  word_count_missing
content_type                                                  
comparison article    697             0.00                0.00
feedly article       2096             1.00                0.00
keyword article     27207             0.01                0.28


In [4]:
# FACT 4 - the rate columns really are x100 percentages, and avg_position==0 means "no data".
print("FACT 4 units:")
print(f"  ctr median = {raw['ctr'].median():.3f}  (0.76 means 0.76%, not 76%)")
print(f"  avg_position == 0 rows (no position data): {(raw['avg_position'] == 0).sum():,}")
print(f"  scroll_rate max = {raw['scroll_rate'].max():.1f}  (can exceed 100 by design)")

# FACT 5 - window separation: features never touch the outcome window.
feats_touch_outcome = [c for c in NUM_FEATURES + CAT_FEATURES
                       if c.endswith("_last_30d") or c in ("trend_pct", "trend_direction")]
print(f"FACT 5 window separation: feature columns that touch the outcome window = {feats_touch_outcome or 'none'}")

FACT 4 units:
  ctr median = 0.070  (0.76 means 0.76%, not 76%)
  avg_position == 0 rows (no position data): 1,205
  scroll_rate max = 300.0  (can exceed 100 by design)
FACT 5 window separation: feature columns that touch the outcome window = none


### 3b. The one thing the contract cannot fix

The two 30-day windows are the smallest honest unit this slice offers. A page whose traffic simply has
weekly seasonality will look like it "declined" every other window. That is a property of the data, and
it is why the paper reports ranking quality with the base rate attached, never a point forecast.

In [5]:
metrics = {
    "task": "ml-04",
    "data": "data/raw/content_refresh_anonymized.csv",
    "rows": int(len(raw)),
    "columns": int(raw.shape[1]),
    "clients": int(raw["client_id"].nunique()),
    "duplicate_content_ids": dupes,
    "decision_window": "prior-30 (days 31-60)",
    "outcome_window": "later-30 (days 1-30)",
    "pool_rule": f"impressions_prev_30d >= {MIN_PREV_IMPRESSIONS}",
    "pool_rows": int(len(pool)),
    "pool_clients": int(pool["client_id"].nunique()),
    "label": "declined_next_30d = impressions_last_30d < 0.8 * impressions_prev_30d",
    "feature_columns": int(X.shape[1]),
    "excluded_overlapping_columns": list(excluded),
}
(OUT / "w03_data_contract_metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
print("receipt written: work/outputs/w03_data_contract_metrics.json")

receipt written: work/outputs/w03_data_contract_metrics.json


## 4. Data limits

- **One snapshot, two windows.** No longer history per page; seasonality cannot be separated from trend.
- **Trailing-90-day aggregation.** Engagement columns can be sparse or geometrically odd (`scroll_rate`,
  `ai_traffic_pct` can exceed 100) — they are context, not clean rates.
- **Pseudonymity.** No client names, domains, URLs, or queries — which is exactly why the work is safe
  to publish, and why "why did this page fall?" can only ever be answered by a human reading the page.
- **Selection by design.** Pages below 100 prior-30 impressions are out of scope: the queue is about
  pages that are *already being seen*.

In [6]:
print(f"pages excluded from the pool by the visibility floor: {len(raw) - len(pool):,}")
print(f"share of slice kept: {100*len(pool)/len(raw):.1f}%")
print("pool covers", f"{pool['client_id'].nunique()} of {raw['client_id'].nunique()} clients.")

pages excluded from the pool by the visibility floor: 11,990
share of slice kept: 60.0%
pool covers 30 of 32 clients.


## Self-check

- [x] Unit of analysis and both time windows stated
- [x] Every field sorted into feature / label / context / excluded
- [x] Each contract claim carries a check
- [x] Data limits named honestly (one snapshot, seasonal confound, pseudonymity)
- [x] Receipt written to `work/outputs/w03_data_contract_metrics.json`
- [x] Committed under `work/notebooks/`